# 99 · Utilidades de desarrollo (borrado con confirmación)

Celdas para reiniciar capas durante el desarrollo y antes de un reproceso completo. **No forma
parte del job.** Cada celda que borra llama a `exigir_confirmacion()`: solo se ejecuta si el
widget `confirmar` contiene exactamente el nombre del catálogo. Un "Run all" con el widget vacío
se detiene en la primera celda de borrado sin tocar nada.

| Celda | Qué borra | Qué conserva |
|---|---|---|
| Reinicio de plata y oro | `ais_posiciones`, `ais_cuarentena`, `calidad_metricas`, checkpoint de plata, todo `gold` | Bronce, landing, dimensiones de `silver` |
| Reinicio total (solo desarrollo) | Lo anterior + bronce, su checkpoint y esquema de Auto Loader, archivos de `entrantes/` | `_descargas/` (no se vuelve a bajar de NOAA), dimensiones |

In [0]:
import config

spark = config.obtener_spark()
w = config.obtener_workspace()

CATALOGO = config.leer_parametro("catalogo", config.CATALOGO_DEFECTO)
CONFIRMAR = config.leer_parametro("confirmar", "")
cfg = config.Config(CATALOGO)
print(f"Catálogo: {cfg.catalogo} · confirmar: '{CONFIRMAR}'")


def exigir_confirmacion() -> None:
    """Detiene la celda salvo que `confirmar` diga exactamente el nombre del catálogo."""
    if CONFIRMAR != cfg.catalogo:
        raise RuntimeError(
            f"Celda de borrado bloqueada. Para ejecutarla, escribe '{cfg.catalogo}' en el widget "
            "`confirmar` y vuelve a correr la celda de parámetros."
        )


def borrar_carpeta(ruta: str) -> None:
    """Borra una carpeta de un Volume con todo su contenido. No falla si no existe."""
    try:
        entradas = list(w.files.list_directory_contents(ruta))
    except Exception:
        return
    for e in entradas:
        borrar_carpeta(e.path) if e.is_directory else w.files.delete(e.path)
    w.files.delete_directory(ruta)
    print(f"  carpeta borrada: {ruta}")


def vaciar_esquema(capa: str) -> None:
    """Borra todas las tablas y vistas de un esquema (el esquema queda, vacío)."""
    objetos = spark.sql(f"""
        SELECT table_name, table_type FROM {cfg.catalogo}.information_schema.tables
        WHERE table_schema = '{capa}'
    """).collect()
    for o in objetos:
        tipo = "VIEW" if o.table_type == "VIEW" else "TABLE"
        spark.sql(f"DROP {tipo} IF EXISTS {cfg.tabla(capa, o.table_name)}")
        print(f"  {tipo.lower()} borrada: {cfg.tabla(capa, o.table_name)}")


def reiniciar_plata_y_oro() -> None:
    for t in ["ais_posiciones", "ais_cuarentena", "calidad_metricas"]:
        spark.sql(f"DROP TABLE IF EXISTS {cfg.tabla('silver', t)}")
        print(f"  tabla borrada: {cfg.tabla('silver', t)}")
    borrar_carpeta(cfg.ruta_checkpoint("silver"))
    vaciar_esquema("gold")

In [0]:
# Reinicio de plata y oro: el siguiente 03_silver reprocesa todo bronce desde cero.
exigir_confirmacion()
reiniciar_plata_y_oro()
print("Plata y oro reiniciados. Las dimensiones de silver (dim_*) se conservan.")

In [0]:
# Reinicio total (solo desarrollo): vuelve el pipeline al estado posterior a 00_preparacion.
exigir_confirmacion()
reiniciar_plata_y_oro()
spark.sql(f"DROP TABLE IF EXISTS {cfg.tabla('bronze', 'ais_posiciones_raw')}")
print(f"  tabla borrada: {cfg.tabla('bronze', 'ais_posiciones_raw')}")
borrar_carpeta(cfg.ruta_checkpoint("bronze"))
borrar_carpeta(cfg.ruta_esquema_autoloader("bronze"))
for e in list(w.files.list_directory_contents(cfg.ruta_ais_entrantes)):
    w.files.delete(e.path)
    print(f"  archivo borrado: {e.path}")
print("Reinicio total listo. _descargas/ y las dimensiones se conservan.")

In [0]:
spark.sql(f"""
    SELECT table_schema, table_name, table_type
    FROM {cfg.catalogo}.information_schema.tables
    WHERE table_schema NOT IN ('information_schema')
    ORDER BY table_schema, table_name
""").show(100, truncate=False)
